# Credit Card Fraud Detection

### Overview

Banks process millions of card payments daily, a tiny fraction of which are fraudulent. The goal of this project is a model that distinguishes fraudulent transactions from normal ones by learning patterns from past examples.

Fraud is rare — roughly 2 in every 1,000 transactions. Because of this, plain accuracy is misleading: always predicting "not fraud" scores 99.8% while catching no fraud at all. The right measure is how many real frauds the model actually catches.

### Steps taken

1. Load the data.
2. Explore the data to understand it.
3. Prepare the data for modelling.
4. Train a simple baseline model for comparison.
5. Train a stronger model (XGBoost) and compare.
6. Evaluate with metrics suited to rare events.
7. Explain the model's behaviour.

---

## Where the data comes from

The dataset is the **Credit Card Fraud Detection** dataset, created from real (anonymized) transactions by a research group at the Université Libre de Bruxelles (ULB) together with the payments company Worldline, and used in published research. It was obtained from **Kaggle**, a well-known platform that hosts public datasets.

The notebook connects to Kaggle once using a small key file (`kaggle.json`) and downloads the dataset automatically. The large data file does not need to be downloaded manually.

**How to get the key file (one time):**
1. Go to https://www.kaggle.com/settings
2. Scroll down to the **API** section.
3. Click **Create New API Token**.
4. The `kaggle.json` file downloads automatically to the **Downloads** folder.

Run the cell below: when the **Choose Files** button appears, select `kaggle.json` from the Downloads folder. The dataset then downloads automatically.

In [ ]:
# This downloads the dataset from Kaggle so the notebook can use it.
# When it runs, click 'Choose Files' and select the kaggle.json key.
from google.colab import files
files.upload()  # pick kaggle.json

!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle
!kaggle datasets download -d mlg-ulb/creditcardfraud
!unzip -o creditcardfraud.zip
print('The data file creditcard.csv is ready.')

## Step 0 — Libraries

Most libraries ship with Colab; two extras are installed below, then all are imported.

In [ ]:
# Install the two libraries not pre-installed in Colab
!pip install -q imbalanced-learn shap

In [ ]:
# Import the libraries used below
import pandas as pd              # data tables
import numpy as np               # numerical operations
import matplotlib.pyplot as plt  # plotting
import seaborn as sns            # statistical plots

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap

# A fixed 'random seed' makes the results reproducible on every run.
SEED = 42
print('Libraries loaded.')

## Step 1 — Load the data

The data file is read into a table (a *DataFrame*), where each row is one transaction. The first few rows are shown below.

In [ ]:
data = pd.read_csv('creditcard.csv')

print('The data has', data.shape[0], 'transactions and', data.shape[1], 'columns.')
data.head()

**The columns:**
- `V1` to `V28` are numeric features describing each transaction. For privacy, their real meaning was hidden behind a math transformation, so they are anonymous numbers.
- `Amount` is the transaction value.
- `Time` is when it happened.
- `Class` is the target to predict: **0 means normal, 1 means fraud.**

## Step 2 — Explore the data

Understanding the data comes before modelling. The key question here: how rare is fraud?

In [ ]:
# Count normal vs fraud
normal = (data['Class'] == 0).sum()
fraud = (data['Class'] == 1).sum()

print('Normal transactions:', normal)
print('Fraud transactions: ', fraud)
print('Fraud is only {:.2f}% of all transactions.'.format(fraud / len(data) * 100))

In [ ]:
# Draw it as a bar chart so the imbalance is easy to see
sns.countplot(x='Class', data=data)
plt.title('Almost all transactions are normal (0), very few are fraud (1)')
plt.savefig('class_imbalance.png', bbox_inches='tight', dpi=150)  # saves the image file
plt.show()

The chart makes the problem clear: fraud is a tiny sliver of the data. This imbalance is handled carefully in the modelling steps.

## Step 3 — Prepare the data

Two steps here:
1. **Put `Amount` and `Time` on a similar scale** to the other columns, so no single column overpowers the model just because its numbers are bigger.
2. **Split the data into two parts:** one part to train the model and a separate part to test it. Testing must use data the model has never seen — otherwise the evaluation is unreliable, like giving a student the exam answers in advance.

In [ ]:
# 1. Scale Amount and Time
scaler = StandardScaler()
data['Amount'] = scaler.fit_transform(data[['Amount']])
data['Time'] = scaler.fit_transform(data[['Time']])

# 2. Separate the features (X) from the target (y)
X = data.drop('Class', axis=1)   # all columns except the target
y = data['Class']                # the target: fraud or not

# 3. Split into training (75%) and testing (25%) sets.
#    stratify=y keeps the same fraud ratio in both parts.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

print('Training transactions:', len(X_train))
print('Testing transactions: ', len(X_test))

## Step 4 — A simple model first (the baseline)

Rather than starting with a complex model, a simple, well-understood one — **Logistic Regression** — is trained first. It sets a baseline score: the bar the stronger model must beat to be worth using.

The setting `class_weight='balanced'` makes it pay extra attention to the rare fraud cases.

In [ ]:
# Train the baseline model
simple_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED)
simple_model.fit(X_train, y_train)

# Predict on the test set
simple_predictions = simple_model.predict(X_test)

print('Results of the baseline model (Logistic Regression):')
print(classification_report(y_test, simple_predictions))

**How to read this:** look at the row for class **1** (fraud). *Recall* is the fraction of all real frauds that were caught — the metric that matters most for a bank.

## Step 5 — A stronger model (XGBoost)

**XGBoost** is one of the most trusted models for tabular data and is widely used in industry and competitions. It builds many small decision trees that work together, each correcting the errors of the previous ones.

The setting `scale_pos_weight` makes it take the rare fraud cases seriously.

In [ ]:
# Ratio of normal to fraud transactions, used to weight the rare class
weight = (y_train == 0).sum() / (y_train == 1).sum()

# Train the stronger model
strong_model = xgb.XGBClassifier(
    n_estimators=300,          # number of trees
    max_depth=4,               # how deep each tree can go
    learning_rate=0.1,         # how fast it learns
    scale_pos_weight=weight,   # pay attention to rare fraud
    eval_metric='aucpr',
    random_state=SEED
)
strong_model.fit(X_train, y_train)

strong_predictions = strong_model.predict(X_test)

print('Results of the stronger model (XGBoost):')
print(classification_report(y_test, strong_predictions))

## Step 6 — Evaluation

Because fraud is rare, evaluation uses **PR-AUC** (higher is better, max is 1.0), a score suited to rare cases. Both models are compared, followed by a **confusion matrix** — a table showing what the stronger model got right and wrong.

In [ ]:
# PR-AUC for each model (uses the model's confidence, not just yes/no)
simple_score = average_precision_score(y_test, simple_model.predict_proba(X_test)[:, 1])
strong_score = average_precision_score(y_test, strong_model.predict_proba(X_test)[:, 1])

print('PR-AUC (higher is better):')
print('  Simple model (Logistic Regression):', round(simple_score, 3))
print('  Strong model (XGBoost):            ', round(strong_score, 3))

In [ ]:
# Confusion matrix for the strong model
matrix = confusion_matrix(y_test, strong_predictions)
sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Fraud'], yticklabels=['Normal', 'Fraud'])
plt.xlabel('What the model predicted')
plt.ylabel('The real answer')
plt.title('XGBoost: predictions vs reality')
plt.savefig('confusion_matrix.png', bbox_inches='tight', dpi=150)  # saves the image file
plt.show()

print('Top-left     = normal, correctly classified as normal')
print('Bottom-right = fraud, correctly caught')
print('Bottom-left  = fraud missed by the model (the costly errors)')

## Step 7 — Model explainability

In banking, a correct prediction is not enough — the reason a transaction was flagged must be **explainable**. **SHAP** shows which features pushed the model's decisions the most.

In [ ]:
# Use a small sample so this runs quickly
sample = X_test.sample(2000, random_state=SEED)
explainer = shap.TreeExplainer(strong_model)
shap_values = explainer.shap_values(sample)

# Each dot is a transaction; features at the top mattered most to the model.
shap.summary_plot(shap_values, sample, show=False)
plt.savefig('shap_summary.png', bbox_inches='tight', dpi=150)  # saves the image file
plt.show()